# 08 生成模块与提示词工程第二部分：Ollama 生成 + 端到端流水线

> 本 notebook **贯穿阶段 0–6**，随开发进度增量追加 C0→C7（见 [`schedule.md`](../schedule.md)「开发与备份工作流」）。
>
> **每完成一个开发阶段**：运行本阶段对应 cell → 更新 `schedule.md` + 根目录 `README.md` → git 提交 → 再进入下一阶段编码。

---

## 章节与阶段对照（增量追加）

| 章节 | 开发阶段 | 内容 | 状态 |
|------|----------|------|------|
| **C0** | 0 环境与骨架 | 路径、`sys.path`、上游 import、Ollama 配置 | ✅ 已完成 |
| **C1** | 1 LLMGenerator | `health_check` + 单条 `generate()` | ✅ 已完成 |
| **C2** | 2 JSON 工具 | `extract_json` / `repair_json` | ✅ 已完成 |
| **C3** | 3 Pipeline | 06→07 联调 | ✅ 已完成 |
| **C4** | 3 Pipeline | 最小路径 `run()` | ✅ 已完成 |
| **C5** | 3 Pipeline | 完整 pipeline + optional stages | ✅ 已完成 |
| **C6** | 4 后处理 | `sources`、引用、免责声明 | ✅ 已完成 |
| **C7** | 5 评测快照 | 批量 query + `generation_eval.json` | ✅ 已完成 |
| C7（复跑） | 6 测试与交付 | 全量复跑 + pytest 摘要 | 🔄 下一步 |

> 内核 **`med-rag-verify`**。Ollama 日常启动见 **C0**；**安装 / 拉模型 / PATH 排错** 见 [`schedule.md`](../schedule.md)「Windows Ollama 环境（方式 A）」。

## C0：环境与路径（阶段 0）

1. 解析路径并 smoke import 上游 06/07；
2. 探测 Ollama 是否在线且已有 `deepseek-r1:7b`。

**日常启动 Ollama（方式 A）**：开始菜单或托盘打开 **Ollama** → 运行下方两个 code cell。若探测失败，见 [`schedule.md` § Windows Ollama 环境](../schedule.md)。

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

# 确保可 import 本阶段 bootstrap（notebook 位于 08/notebooks）
_nb_dir = Path.cwd().resolve()
if _nb_dir.name == "notebooks":
    _stage08_src = _nb_dir.parent / "src"
else:
    _stage08_src = _nb_dir / "08 生成模块与提示词工程第二部分" / "src"
if str(_stage08_src) not in sys.path:
    sys.path.insert(0, str(_stage08_src))

from bootstrap import OLLAMA_BASE_URL, OLLAMA_MODEL, bootstrap_paths

paths = bootstrap_paths(_nb_dir)
ROOT = paths["root"]
STAGE08 = paths["stage08"]
STAGE06 = paths["stage06"]
STAGE07 = paths["stage07"]

print("ROOT:", ROOT)
print("STAGE08:", STAGE08)
print("Ollama:", OLLAMA_BASE_URL, "| model:", OLLAMA_MODEL)

# 上游 smoke import（05 由 06 pipeline 按需挂载，此处不加入 sys.path）
from pipeline import RetrievalPipeline  # noqa: E402  06
from context_assembler import ContextAssembler  # noqa: E402  07
from prompts import PROMPT_STAGES  # noqa: E402  07

print("RetrievalPipeline:", RetrievalPipeline)
print("ContextAssembler:", ContextAssembler)
print("PROMPT_STAGES keys:", list(PROMPT_STAGES.keys()))
print("\n✅ C0 smoke: paths + upstream imports OK")

ROOT: D:\谷歌
STAGE08: D:\谷歌\08 生成模块与提示词工程第二部分
Ollama: http://127.0.0.1:11434 | model: deepseek-r1:7b
RetrievalPipeline: <class 'pipeline.RetrievalPipeline'>
ContextAssembler: <class 'context_assembler.ContextAssembler'>
PROMPT_STAGES keys: ['evidence_evaluator', 'answer_generator', 'critical_reviewer', 'final_assembler']

✅ C0 smoke: paths + upstream imports OK


### C0（续）：Ollama 探测

期望 `Probe OK: True`。失败时见 [`schedule.md`](../schedule.md)「Windows Ollama 环境」。

In [2]:
import httpx

def probe_ollama(base_url: str, model: str) -> dict:
    try:
        with httpx.Client(timeout=5.0) as client:
            resp = client.get(f"{base_url}/api/tags")
            resp.raise_for_status()
            names = [m.get("name", "") for m in resp.json().get("models", [])]
    except httpx.HTTPError as exc:
        return {"ok": False, "error": str(exc), "models": [], "reachable": False}
    has_model = any(model in n for n in names)
    return {
        "ok": has_model,
        "reachable": True,
        "models": names,
        "error": None if has_model else f"model {model!r} not in /api/tags",
    }

status = probe_ollama(OLLAMA_BASE_URL, OLLAMA_MODEL)
print("Ollama URL:", OLLAMA_BASE_URL)
print("Service reachable:", status.get("reachable", False))
print("Probe OK (model ready):", status["ok"])
if status["models"]:
    print("Installed models:", status["models"])

if not status.get("reachable"):
    print("\n⚠️ 无法连接 → 托盘/开始菜单启动 Ollama，详见 schedule.md")
elif status["error"]:
    print("\nNote:", status["error"])
    print("拉模型见 schedule.md「Windows Ollama 环境」")
else:
    print(f"\n✅ C0 Ollama OK — {OLLAMA_MODEL!r} available")

Ollama URL: http://127.0.0.1:11434
Service reachable: True
Probe OK (model ready): True
Installed models: ['deepseek-r1:7b']

✅ C0 Ollama OK — 'deepseek-r1:7b' available


## C1：LLMGenerator smoke（阶段 1）

`health_check()` + 单条 `generate()`；默认 `think=False`（01 阶段结论，避免 deepseek-r1 耗尽在 thinking）。

In [3]:
import time

from llm_generator import LLMGenerator

llm = LLMGenerator(model_name=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, timeout=180.0)

print("health_check:", llm.health_check())

t0 = time.perf_counter()
answer = llm.generate(
    "In one sentence, what is metformin used for?",
    system_prompt="You are a concise medical assistant. Answer in English.",
    temperature=0.2,
    max_tokens=512,  # deepseek-r1 需留足预算，避免全耗在 thinking
    think=False,
)
elapsed = time.perf_counter() - t0

preview = answer[:400] + ("..." if len(answer) > 400 else "")
print(f"\nElapsed: {elapsed:.1f}s | chars: {len(answer)}")
print("Response preview:\n", preview or "(empty — 尝试增大 max_tokens)")
print("\n✅ C1 LLMGenerator smoke OK" if answer.strip() else "\n⚠️ 空响应，见 schedule / 01 阶段 think 说明")

health_check: True

Elapsed: 8.8s | chars: 191
Response preview:
 Metformin is primarily used to manage blood glucose levels in individuals with type 2 diabetes by enhancing insulin sensitivity and aiding in insulin production, thereby lowering blood sugar.

✅ C1 LLMGenerator smoke OK


## C2：JSON 提取与修复（阶段 2）

围栏剥离、`repair_json`、证据评估 schema、`generate_json()` 联调（含 mock 字符串，无需额外 LLM 调用）。

In [4]:
from json_utils import (
    extract_json,
    filter_chunks_by_evidence_eval,
    parse_evidence_evaluation,
    repair_json,
)

# 1) markdown 围栏
fenced = '''Analysis:
```json
{"relevant_chunk_ids": ["PMC520826"], "excluded_chunk_ids": [], "notes": "strong RCT"}
```'''
print("1 fenced:", parse_evidence_evaluation(fenced))

# 2) 残缺 JSON（缺右括号）
broken = '{"relevant_chunk_ids": ["PMC1"], "excluded_chunk_ids": ["PMC9"], "notes": "drop noisy"'
print("2 repaired:", extract_json(repair_json(broken)))

# 3) 降级：解析失败 → 不筛选
chunks = [{"chunk_id": "PMC1"}, {"chunk_id": "PMC2"}]
print("3 no filter:", filter_chunks_by_evidence_eval(chunks, None))
print("3 relevant only:", filter_chunks_by_evidence_eval(chunks, parse_evidence_evaluation(fenced)))

# 4) generate_json 联调（mock 模型输出，不调用 Ollama）
mock_llm_output = 'Sure.\n```json\n{"relevant_chunk_ids": [], "excluded_chunk_ids": ["PMC9"], "notes": "irrelevant"}\n```'
print("4 mock parse:", parse_evidence_evaluation(mock_llm_output))
print("4 filtered:", filter_chunks_by_evidence_eval(chunks, parse_evidence_evaluation(mock_llm_output)))

print("\n✅ C2 JSON utils OK")

1 fenced: {'relevant_chunk_ids': ['PMC520826'], 'excluded_chunk_ids': [], 'notes': 'strong RCT'}
2 repaired: {'relevant_chunk_ids': ['PMC1'], 'excluded_chunk_ids': ['PMC9'], 'notes': 'drop noisy'}
3 no filter: [{'chunk_id': 'PMC1'}, {'chunk_id': 'PMC2'}]
3 relevant only: [{'chunk_id': 'PMC1'}, {'chunk_id': 'PMC2'}]
4 mock parse: {'relevant_chunk_ids': [], 'excluded_chunk_ids': ['PMC9'], 'notes': 'irrelevant'}
4 filtered: [{'chunk_id': 'PMC1'}, {'chunk_id': 'PMC2'}]

✅ C2 JSON utils OK


## C3：06→07 联调（阶段 3）

这里先不跑完整生成，只验证：

1. 能读取 06 离线评测样例（`pipeline_eval.json`）
2. 能把 `reranked` 候选交给 07 `ContextAssembler`
3. 能拿到 `context_text` / `metadata`

In [5]:
import json

PIPELINE_EVAL_PATH = ROOT / "06 检索系统开发第二部分" / "outputs" / "samples" / "pipeline_eval.json"
with PIPELINE_EVAL_PATH.open("r", encoding="utf-8") as f:
    eval_payload = json.load(f)

query_item = eval_payload["queries"][0]
query_text = query_item["query"]
candidates = query_item["reranked"]

assembler = ContextAssembler(tokenizer_name=None)
assembled_c3 = assembler.assemble(candidates, max_context_tokens=1200)

print("query:", query_text)
print("candidates:", len(candidates))
print("selected:", len(assembled_c3.selected_chunks))
print("estimated_tokens:", assembled_c3.metadata.estimated_tokens)
print("context preview:\n", assembled_c3.context_text[:500], "...")

query: metformin cardiovascular effects
candidates: 5
selected: 3
estimated_tokens: 1200
context preview:
 Daily rhythms in plasma levels of homocysteine

There is accumulated evidence that plasma concentration of the sulfur-containing amino-acid homocysteine (Hcy) is a prognostic marker for cardiovascular morbidity and mortality. Both fasting levels of Hcy and post methionine loading levels are used as prognostic markers. The aim of the present study was to investigate the existence of a daily rhythm in plasma Hcy under strictly controlled nutritional and sleep-wake conditions. We also investigated  ...


## C4：最小路径 `MedicalGenerationPipeline.run()`

最小路径：`skip_evidence_eval=True`、`skip_critical_review=True`。

In [6]:
from generation_pipeline import MedicalGenerationPipeline


class OfflineRetrievalAdapter:
    """用 06 离线样例模拟 retrieval_pipeline.run()。"""

    def __init__(self, query_payload: dict):
        self.query_payload = query_payload

    def run(self, query: str) -> dict:
        return {
            "query": query,
            "retrieval": self.query_payload.get("retrieval", {"fused": []}),
            "reranked": self.query_payload.get("reranked", []),
        }


offline_retrieval = OfflineRetrievalAdapter(query_item)
llm = LLMGenerator(model_name=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, timeout=180.0)

pipe_min = MedicalGenerationPipeline(
    retrieval_pipeline=offline_retrieval,
    context_assembler=assembler,
    llm_generator=llm,
    skip_evidence_eval=True,
    skip_critical_review=True,
)

result_min = pipe_min.run(query_text)
print("answer preview:\n", result_min["answer"][:600], "...")
print("\nstage_success:", result_min["generation_metrics"]["stage_success"])
print("sources:", result_min["sources"][:3])

answer preview:
 **Answer:**

Metformin's impact on cardiovascular risks can be summarized as follows:

1. **Homoysteine (Hcy) Levels:** A study revealed a daily rhythm in Hcy concentrations under controlled conditions, with midday nadir and nighttime peak. However, metformin did not demonstrate different effects during morning or evening loading.

2. **Lipid Profile Management:**
   - A study showed that Metformin influences lipid profiles, particularly increasing HDL-c (high-density lipoprotein cholesterol) and lowering total cholesterol (TC). The NVP group exhibited a larger increase in HDL-c compared to th ...

stage_success: {'assemble': True, 'evidence_eval': True, 'draft': True, 'review': True, 'final': True, 'postprocess': True}
sources: [{'index': 1, 'chunk_id': 'PMC520826', 'source_title': 'Daily rhythms in plasma levels of homocysteine', 'doc_id': 'PMC520826', 'relevance_score': 0.298591}, {'index': 2, 'chunk_id': 'PMC521687', 'source_title': 'Factors influencing preoperativ

## C5：完整 pipeline（含 optional stages）

启用证据评估与批判审查，对比最小路径结果。

In [7]:
pipe_full = MedicalGenerationPipeline(
    retrieval_pipeline=offline_retrieval,
    context_assembler=assembler,
    llm_generator=llm,
    skip_evidence_eval=False,
    skip_critical_review=False,
)

result_full = pipe_full.run(query_text)
print("answer preview:\n", result_full["answer"][:600], "...")
print("\nstage_success:", result_full["generation_metrics"]["stage_success"])
print("evidence_evaluation:", result_full["intermediate_results"]["evidence_evaluation"])
print("review_feedback preview:", (result_full["intermediate_results"]["review_feedback"] or "")[:200])

answer preview:
 **Answer:**

- **Overview:** The provided context discusses various studies on cardiovascular risk factors, including homocysteine levels, preoperative stress responses, dyslipidemia, and antihypertensive drug effects. However, there is no direct evidence linking metformin to specific cardiovascular effects such as changes in homocysteine (Hcy) levels or antihypertensive responses.

- **Specific Findings:**
  - The study on Hcy investigates its role as a prognostic marker for cardiovascular morbidity and mortality but does not involve metformin.
  - The preoperative stress response study focus ...

stage_success: {'assemble': True, 'evidence_eval': True, 'draft': True, 'review': True, 'final': True, 'postprocess': True}
evidence_evaluation: {'relevant_chunk_ids': [], 'excluded_chunk_ids': [], 'notes': ''}
review_feedback preview: ### **Medical Safety Review of the Provided Context**

#### **1. Overclaims:**
- **Claim:** "There is accumulated evidence that plasma concen

In [8]:
# C5 小对比：最小路径 vs 完整路径
cmp = {
    "min_answer_len": len(result_min["answer"]),
    "full_answer_len": len(result_full["answer"]),
    "min_sources": len(result_min["sources"]),
    "full_sources": len(result_full["sources"]),
    "min_total_s": result_min["generation_metrics"]["total_time_seconds"],
    "full_total_s": result_full["generation_metrics"]["total_time_seconds"],
}
cmp

{'min_answer_len': 2129,
 'full_answer_len': 2430,
 'min_sources': 5,
 'full_sources': 5,
 'min_total_s': 24.848,
 'full_total_s': 38.792}

## C6：后处理与来源对齐（阶段 4）

展示阶段 4 三个核心点：

1. `answer` 中引用标记（如 `[1] [2]`）
2. `sources` 列表（`chunk_id/source_title/doc_id/relevance_score`）
3. 固定医学免责声明

In [9]:
from postprocess import MEDICAL_DISCLAIMER

print("answer preview:\n")
print(result_full["answer"][:1200])

print("\n---- checks ----")
print("has ref marker:", "[1]" in result_full["answer"])
print("has sources block:", "Sources:" in result_full["answer"])
print("has disclaimer:", MEDICAL_DISCLAIMER in result_full["answer"])

print("\nfirst 3 sources:")
for src in result_full["sources"][:3]:
    print(src)

answer preview:

**Answer:**

- **Overview:** The provided context discusses various studies on cardiovascular risk factors, including homocysteine levels, preoperative stress responses, dyslipidemia, and antihypertensive drug effects. However, there is no direct evidence linking metformin to specific cardiovascular effects such as changes in homocysteine (Hcy) levels or antihypertensive responses.

- **Specific Findings:**
  - The study on Hcy investigates its role as a prognostic marker for cardiovascular morbidity and mortality but does not involve metformin.
  - The preoperative stress response study focuses on norepinephrine levels before induction of anaesthesia using clonidine, which is unrelated to metformin.
  - The dyslipidemia study explores the effects of polymorphisms in lipid metabolism genes on antihypertensive treatment responses but does not involve metformin.

- **Constraints:** None of the studies provided directly address how metformin influences cardiovascular outc

## C7：批量评测与快照导出（阶段 5）

运行 CLI 脚本批量处理固定 query，导出：

- `outputs/samples/generation_eval.json`
- `outputs/logs/generation_eval_*.json`

In [11]:
import json
import subprocess
import sys

script_path = STAGE08 / "scripts" / "run_generation_eval.py"
cmd = [sys.executable, str(script_path)]
print("running:", " ".join(cmd))

# 用 stage08 作为工作目录，避免相对路径导致 exit status 2
proc = subprocess.run(
    cmd,
    check=False,
    cwd=str(STAGE08),
    capture_output=True,
    text=True,
)

if proc.returncode != 0:
    print("\n[STDOUT]\n", proc.stdout)
    print("\n[STDERR]\n", proc.stderr)
    raise RuntimeError(f"run_generation_eval failed with exit code {proc.returncode}")

print(proc.stdout)

sample_path = STAGE08 / "outputs" / "samples" / "generation_eval.json"
report = json.loads(sample_path.read_text(encoding="utf-8"))

print("\nquery_count:", report["query_count"])
for item in report["queries"]:
    m = item["generation_metrics"]
    print(
        f"- {item['query']} | total={m['total_time_seconds']}s | "
        f"answer_len={len(item['answer'])} | sources={len(item['sources'])}"
    )

print("\n✅ C7 generation eval exported:", sample_path)

running: c:\Users\10138\miniconda3\envs\med-rag-verify\python.exe D:\谷歌\08 生成模块与提示词工程第二部分\scripts\run_generation_eval.py
[OK] What is the treatment for MI? | total=34.538s | answer_len=1770 | sources=3
[OK] metformin cardiovascular effects | total=32.247s | answer_len=2050 | sources=3
[OK] papers on malaria after 2015 | total=47.347s | answer_len=2910 | sources=4
[OK] warfarin atrial fibrillation elderly | total=44.16s | answer_len=3166 | sources=3

Saved sample: D:\谷歌\08 生成模块与提示词工程第二部分\outputs\samples\generation_eval.json
Saved log:    D:\谷歌\08 生成模块与提示词工程第二部分\outputs\logs\generation_eval_20260701_173000.json


query_count: 4
- What is the treatment for MI? | total=34.538s | answer_len=1770 | sources=3
- metformin cardiovascular effects | total=32.247s | answer_len=2050 | sources=3
- papers on malaria after 2015 | total=47.347s | answer_len=2910 | sources=4
- warfarin atrial fibrillation elderly | total=44.16s | answer_len=3166 | sources=3

✅ C7 generation eval exported: D:\谷歌\08 生成模块与

In [12]:
# C7 汇总视图（可复制到报告）
summary = {
    "query_count": report["query_count"],
    "avg_total_s": round(
        sum(q["generation_metrics"]["total_time_seconds"] for q in report["queries"]) / report["query_count"],
        3,
    ),
    "avg_answer_len": round(
        sum(len(q["answer"]) for q in report["queries"]) / report["query_count"],
        1,
    ),
}
summary

{'query_count': 4, 'avg_total_s': 39.573, 'avg_answer_len': 2474.0}

## C7（复跑确认）：pytest 摘要（阶段 6）

阶段 6 要求回归确认。这里在 notebook 内触发 `pytest tests/ -v`，并打印最后摘要行。

In [ ]:
import subprocess
import sys

cmd = [sys.executable, "-m", "pytest", "tests/", "-v"]
proc = subprocess.run(cmd, cwd=str(STAGE08), check=False, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(f"pytest failed with code {proc.returncode}")

summary_line = ""
for line in proc.stdout.splitlines()[::-1]:
    if "passed" in line and "=" in line:
        summary_line = line
        break
print("\npytest summary:", summary_line or "(not found)")